# **Import**

In [ ]:
# Install required packages
%pip install -q datasets
%pip install -q transformers
%pip install -q sentence-transformers


import os
import sys
from datasets import load_dataset
import json
from transformers import pipeline, set_seed
import random
import torch
import tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# **Seed to ensure reproducibility**

In [11]:
def set_seeds(seed=42):

    random.seed(seed)
    # In caso dovessimo usare Numpy
    # np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
        
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    # Seed on transformers library
    set_seed(seed)
    

set_seeds(seed=42)

# **Environment Setup**

In [12]:
# Automatic Environment Detection
if 'google.colab' in sys.modules:
    print("🌐 Environment detected: Google Colab")
    
    # Automatically mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    BASE_DIR = '/content/drive/MyDrive/MNLP_2026_HW2_2278125_2283937/' 

else:
    print("💻 Environment detected: Local Machine")
    
    BASE_DIR = 'C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026' 

# Defining paths for subdirectories 
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
PLOTS_DIR = os.path.join(BASE_DIR, 'plots')

# Creating directories if they don't exist 
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"📁 Base Directory set to: {BASE_DIR}")

# Set up hardware acceleration (GPU if available, else CPU)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected: {device}")

💻 Environment detected: Local Machine
📁 Base Directory set to: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026
Device selected: cuda


# **Dataset Loading and Exploration**

In [13]:
ds = load_dataset(
    "sapienzanlp-course-materials/hw-mnlp-2026"
)

# Sanity check
for split in ds.keys():
    print(f"Split: {split:6} | Features: {ds[split].column_names}")

# Unpack splits into dedicated variables for easier access
train = ds['train']
test = ds['test']
blind = ds['blind']

Split: test   | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']
Split: blind  | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']
Split: train  | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']


# **Rankings Loading**

In [14]:
# Rankings file name definition
rankings_test_filename = 'hit_k_1-test-distilbert-triplet-cosine_(reranking).jsonl'
rankings_blind_filename = 'hit_k_1-blind-distilbert-triplet-cosine_(reranking).jsonl'

# Complete path creation
rankings_test_path = os.path.join(OUTPUTS_DIR, rankings_test_filename)
rankings_blind_path = os.path.join(OUTPUTS_DIR, rankings_blind_filename)

retrieved_results_test = []
retrieved_results_blind = []

# Test set Rankings loading
try:
    with open(rankings_test_path, 'r', encoding='utf-8') as f:
        for line in f:
            retrieved_results_test.append(json.loads(line.strip()))
            
    print(f"✅ Test Set rankings loaded with success!")
    print(f"   📍 File: {rankings_test_path}")

except FileNotFoundError:
    print(f"❌ ERROR: TEST SET rankings does not found:\n{rankings_test_path}\n")


# Blind set Rankings loading
try:
    with open(rankings_blind_path, 'r', encoding='utf-8') as f:
        for line in f:
            retrieved_results_blind.append(json.loads(line.strip()))
            
    print(f"✅ Blind Set rankings loaded with success!")
    print(f"   📍 File: {rankings_blind_path}")

except FileNotFoundError:
    print(f"❌ ERROR: BLIND SET rankings does not found:\n{rankings_blind_path}")

# Unique Dictionary creation {'query_id1': [rankings], 'query_id2': [rankings], ...}
rankings_test = {}
for item in retrieved_results_test:
    rankings_test.update(item)

rankings_blind = {}
for item in retrieved_results_blind:
    rankings_blind.update(item)

✅ Test Set rankings loaded with success!
   📍 File: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1-test-distilbert-triplet-cosine_(reranking).jsonl
✅ Blind Set rankings loaded with success!
   📍 File: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1-blind-distilbert-triplet-cosine_(reranking).jsonl


# **Small LMs Implementation**

In [ ]:
# Da rendere universale per tutti i settings
# Una risposta alla volta è troppo lento

def answers(dataset, settings='Baseline'):

    answers = {}

    nome_split = str(dataset.split).capitalize() 

    desc = f"{settings} Generation ({nome_split} set)"

    for sample in tqdm.tqdm(dataset, desc= desc):

        query_id = sample['query_id']
        query = sample['query']
        
        # Answer Generation
        output = generator(
            query,
            batch_size=16,
            max_new_tokens=50,       
            return_full_text=False,  
            do_sample = False,       
            pad_token_id=generator.tokenizer.eos_token_id 
        )
        
        generated_answer = output[0]['generated_text'].strip()
        
        answers[query_id] = {
            "query": query,
            "generated_answer": generated_answer
        }

    return answers

## **Llama-3.2-3B-Instruct**

In [34]:
# Da cambiare il nome del modello, meta mi deve accettare la richiesta

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
generator = pipeline(
    "text-generation", 
    model=model_id, 
    dtype=torch.bfloat16, 
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### **Baseline**

In [ ]:
Llama_baseline_test_answers = answers(test)

print(Llama_baseline_test_answers)
    
    

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 Inizio generazione batch (2000 query)...


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

<function answers at 0x0000020A1B90F2E0>


## **LM Colab Friendly**

{'4HZKjht3X13n': {'query': 'where is the lady with an ermine located', 'generated_answer': 'in relation to the castle?  Answer choices: - outside of it. - inside of it. - on top of it. - next to it. - behind it. - near it.\nThe answer is "inside of it." The question asks'}, 'BeCayzNq9ZgD': {'query': 'lok sabha or rajya sabha which is bigger', 'generated_answer': '?\nWhich one is bigger, Lok Sabha or Rajya Sabha?\n\nMulti-choice problem: Are these two questions inquiring about the same information?\nAvailable options:\n (A). no\n (B). yes\n\n(B). Yes.\n\nThese two questions are asking'}, 'ZP7FeO0UVt3U': {'query': 'who was the actress who played pinky tuscadero on happy days', 'generated_answer': '??\n\nTo find out which actress played Pinky Tuscadero on Happy Days, I\'ll follow these steps:\n\n1. Recall information about the show: "Happy Days" is a popular American sitcom that aired from 1974 to 1'}, 'Tt88r7IhbHtg': {'query': 'who wrote the song i want you back', 'generated_answer': '?\